# JN-E — CO Reconciliation (v4 ↔ CKAN), derived & baseline-gated

**What this is.** The Certificate-of-Occupancy reconciliation between our independent reconstruction (v4)
and Berkeley's filed HCD APR (the CKAN mirror) — **every figure DERIVED here**, asserted against an
**external timestamped baseline** (`data/baselines/reconciliation_baseline_*.json`), never hardcoded.

**The text cells are the deliverable.** Each section answers: *where does this data come from* (the flow),
*what transformation happens and what it assumes*, and *what could be wrong* (the verifiability concern).

**Role-discipline contract (the load-bearing invariant).**
DATA FLOW: `raw CPRA xlsx → JN-A events → JN-C classification → gated corrections → THIS reconciliation`.
CKAN **never appears in that chain** — it enters ONLY at the comparison step. **If CKAN ever flows INTO a
derived number, the reconciliation is circular and void.** That single concern governs the whole notebook.

## §1 — Setup + the two sources (read-only)
**Where from.** v4 = the end of our derivation chain (raw CPRA → JN-A → JN-C → corrections). CKAN mirror =
the ORACLE (reconcile-target only). **Verifiability:** both opened read-only and the v4 **sha is pinned** —
the baseline is meaningless unless you reconcile the *same* DB. If the sha drifts, the gate flags it.

In [ ]:
import os, sqlite3, json, glob, hashlib
import pandas as pd
ROOT=os.path.expanduser('~/berkeley-data')
V4=os.path.join(ROOT,'databases','berkeley_housing_v4.db'); HCD=os.path.join(ROOT,'databases','hcd_apr_mirror_2026-06-17_fresh.db')
def ro(p): return sqlite3.connect(f'file:{p}?mode=ro',uri=True)
def v4_sha():
    h=hashlib.sha256()
    with open(V4,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()[:16]
BASE=json.load(open(sorted(glob.glob(os.path.join(ROOT,'data','baselines','reconciliation_baseline_*.json')))[-1]))
print('v4 sha:', v4_sha(), '| baseline as_of:', BASE['as_of'], '| pinned sha:', BASE['v4_sha'])
print('CONTRACT:', BASE['verifiability_contract']['oracle_not_source'])

## §2 — The headline: CO 3,676 vs city 4,022 (−346)
**Where from.** *Our* CO = finaled `new_unit` master permits with `net_units>0` (the ADR-002 verdict layer,
classified in JN-C, corrected by the gated writes). *City* CO = the sum of all 11 `CO_*` income-tier columns
in `table_a2` (the city's curated unit-creating rollup).
**Transformation & assumption.** We compare a **per-permit verdict count** to a **tier-sum rollup**.
**Verifiability concern (grain comparability):** the two grains are comparable because both are
"net new dwelling units that reached completion in Berkeley" — but the comparison is **fragile** where the
city aggregates differently than we attribute per-permit (phased buildings, ADUs, re-platted parcels). Those
fragilities are exactly what the decomposition (§5–§11) isolates. **City CO is NEVER an input — only the target.**

In [ ]:
co = ro(V4).execute("SELECT COALESCE(SUM(x.net_units),0) FROM events e JOIN event_classifications x "
  "ON x.event_id=e.event_id WHERE e.event_type_code='permit_finaled' AND x.housing_role='new_unit' "
  "AND x.is_master=1 AND COALESCE(x.net_units,0)>0").fetchone()[0]
ta=pd.read_sql('SELECT * FROM table_a2', ro(HCD)); cc=[c for c in ta.columns if c.startswith('CO_') and 'DT' not in c]
for c in cc: ta[c]=pd.to_numeric(ta[c],errors='coerce').fillna(0)
city=int(ta[cc].sum().sum()); print(f'OUR CO={co}  CITY CO={city}  GAP={co-city}')

## §3 — The ledger: how 3,066 became 3,676
**Where from.** Each step is a *gated, audited write* to v4 (`docs/audit/2026-06-28..29_*`). The ledger is the
audit trail made runnable — it bridges the original baseline (3,066) to the current derived CO (3,676).
**Verifiability:** the ledger arithmetic is asserted; the per-row provenance points at the audit doc.

In [ ]:
ledger=[('baseline (pre-corrections)',3066),('C2 multifamily count-gap',+1036),('C3 Shattuck phantom-master',-163),
        ('C3 ADU-tail ancillary',-17),('C-multifamily phase-collapse',-199),('dedup47 duplicate-file-row',-47)]
run=0
for name,d in ledger: run+=d; print(f'  {name:34} {d:+6}  -> {run}')
assert run==co, f'ledger {run} != derived CO {co}'
print('ledger reconciles to derived CO:', run)

## §4 — Decomposition framing
The −346 is **not a uniform shortfall** — it is a NET of large offsetting flows. Two directions:
**forward** (city credits, we don't) and **reverse** (we credit, city doesn't). The adjudication rule,
stated once and applied throughout: **the mirror ENUMERATES where we differ; v4 ADJUDICATES each case from
our own evidence. City-silence is never proof.**

## §5 — Permit-mismatch noise (~689, net-zero)  ·  folds `reverse_overcount_triage.py`
**Where from.** The reverse triage of our counted completions the city credits under a *different* permit#/ID
on the same parcel. **Transformation:** match our completions to city credits by APN; the matched-but-
different-ID set is noise. **Verifiability concern — the MATCHING:** a "mismatch" could be a *real*
disagreement if the APN/address join is wrong. We size it by APN + magnitude proximity; it is **approximate**,
which is why ~689 is DOCUMENTED in the baseline but **not hard-gated**. It nets to zero because both sides
count the same unit under different identifiers.

In [ ]:
print('permit-mismatch noise (documented, heuristic):', BASE['documented_not_gated']['permit_mismatch_noise']['value'],
      '-', BASE['documented_not_gated']['permit_mismatch_noise']['sign'])
print('concern:', BASE['documented_not_gated']['permit_mismatch_noise']['verifiability_concern'])

## §6 — Phase-handling, both directions  ·  folds `three_multifam_*`, `c_multifamily_collapse_write.py`
**Assumption (load-bearing): count-once — one building, one count, at the unit-bearing completion phase.**
Foundation/podium/superstructure are *phases of one building*; counting each = double. The systematic finding:
the classifier handled phased multifamily **inconsistently** — sometimes both phases→`new_unit` (over),
sometimes the completion→`ambiguous` (under). **Over** (−199) is already corrected (in §3 ledger, in 3,676);
**under** (+147) is held → §7.

In [ ]:
print('phase OVER corrected (in CO):', BASE['documented_not_gated']['phase_over_corrected']['value'], BASE['documented_not_gated']['phase_over_corrected']['status'])
print('phase UNDER held (NOT in CO):', BASE['documented_not_gated']['phase_under_held']['value'], BASE['documented_not_gated']['phase_under_held']['status'])

## §7 — The +147 held under-count (the SHARPEST verifiability cell)  ·  folds `three_multifam_families/siblings.py`
**Where from.** Three multifamily completion permits classified `ambiguous`: B2021-03302 (+69), B2018-03422
(+55), B2016-05139 (+23). **The concern, stated plainly:** *our* WorkDescriptions carry **NO unit count** for
these — so the ONLY source of 69/55/23 is the **city's** filing. **Adopting +147 would be oracle-as-source —
circular — and is forbidden.** Therefore +147 is **HELD-not-verified**. The ONLY way to verify it independently
is the **Accela / architect-plan harvest** (a separate work stream). Until then it does not enter our CO.

In [ ]:
import re
ev=pd.read_sql("SELECT e.source_record_key sk, json_extract(e.raw_payload,'$.WorkDescription') wd "
  "FROM events e WHERE e.source_record_key IN ('B2021-03302','B2018-03422','B2016-05139')", ro(V4)).drop_duplicates('sk')
for _,r in ev.iterrows():
    has_count = bool(re.search(r'\d+\s*(?:dwelling|residential|rental)?\s*units?', str(r.wd or ''), re.I))
    print(f"  {r.sk}: our-text-has-unit-count={has_count}  | {str(r.wd)[:70]}")
print('=> our text lacks the count; 69/55/23 come ONLY from the city -> HELD-not-verified (+147).')

## §8 — ADU recall (~4)  ·  folds `adu_recall_gap_sizing.py`
**Where from.** Manufactured-home / within-existing-home ADUs held `ambiguous`. **null-not-zero:** a missing
count is unknown, never 0. **Verifiability:** a lower bound — the broader `adu_flag`-nonhousing-role bucket is
not fully sized, so ~4 is DOCUMENTED, not gated.

In [ ]:
print('ADU recall (documented, lower bound):', BASE['documented_not_gated']['adu_recall']['value'])

## §9 — Dedup (already corrected) — and why the count DEPENDS on it  ·  folds dedup47 + event-dedup
**Where from.** The CPRA two-file overlap (2018-2022 + 2023-2025) made some permits emit duplicate
milestone events. **The CO count's correctness DEPENDS on dedup being complete.** dedup47 removed 4 permits'
duplicate *finaled* events (−47, in §3 ledger); the 2,870-event structural dedup was CO-neutral.
**Verifiability — checked:** below we confirm **0 counted permits with >1 finaled-master event** (the dedup
held), and the event-dedup was proven CO-unchanged in `docs/audit/2026-06-29_event_dedup_write.md`.

In [ ]:
dups=ro(V4).execute("SELECT COUNT(*) FROM (SELECT e.source_record_key FROM events e JOIN event_classifications x "
  "ON x.event_id=e.event_id WHERE e.event_type_code='permit_finaled' AND x.housing_role='new_unit' AND x.is_master=1 "
  "AND COALESCE(x.net_units,0)>0 GROUP BY e.source_record_key HAVING COUNT(*)>1)").fetchone()[0]
print('counted permits with >1 finaled-master event:', dups, '(expect 0 -> dedup complete, CO is dedup-clean)')
assert dups==0, 'DEDUP INCOMPLETE — CO count is not trustworthy until this is 0'

## §10 — The BP side, re-established at permit-level: 3,945 (retires 4,911)
**Where from.** FIRST building-permit issuance, **permit-level** = `COUNT(DISTINCT source_record_key)` over
`permit_issued`/`new_unit`/`master`. **The verifiability lesson (deploy-state-decay):** the prior figure
**4,911 was unverifiable** — it had *no runnable provenance* (ad-hoc prose in PROGRESS) and was **event-inflated**
(the cross-file overlap doubled ~1,430 issued events). **A number you cannot re-derive is a number you cannot
trust.** We retire it: derive at permit-level (3,945; event-level 3,946 — they match now that dedup is done).
**Coverage caveat:** tracked-project BPs only, an internal lower bound (the full-city BP stream is unmodeled).

In [ ]:
ev_lvl=ro(V4).execute("SELECT COALESCE(SUM(x.net_units),0) FROM events e JOIN event_classifications x "
  "ON x.event_id=e.event_id WHERE e.event_type_code='permit_issued' AND x.housing_role='new_unit' AND x.is_master=1").fetchone()[0]
pm_lvl=ro(V4).execute("WITH one AS (SELECT e.source_record_key sk, x.net_units nu, "
  "ROW_NUMBER() OVER (PARTITION BY e.source_record_key ORDER BY e.event_id) rn FROM events e "
  "JOIN event_classifications x ON x.event_id=e.event_id WHERE e.event_type_code='permit_issued' "
  "AND x.housing_role='new_unit' AND x.is_master=1) SELECT COALESCE(SUM(nu),0) FROM one WHERE rn=1").fetchone()[0]
print(f'BP-issued units: event-level={ev_lvl}  PERMIT-level={pm_lvl}  (retires the un-provenanced 4,911)')

## §11 — The ~−150 residual (the open, sensitive direction)  ·  folds `structural_gap_triage.py`
**Where from.** After both directions are corrected, the residual is **genuine real-housing-the-city-counts-
that-we-don't** — the forward triage (COVERAGE / DETECTION / TIMING / POSSIBLE_CITY_OVER).
**Verifiability — the sharpest open concern:** this requires **per-permit INDEPENDENT proof** (e.g. the
39-unit congregate CITY_UNDER candidate), **never CKAN-silence**. It is **not closeable by adoption**; it is
DOCUMENTED (~−150), not gated, and is the standing adjudication queue.

In [ ]:
print('residual (documented, open):', BASE['documented_not_gated']['residual']['value'], '-', BASE['documented_not_gated']['residual']['status'])
print('concern:', BASE['documented_not_gated']['residual']['verifiability_concern'])

## §12 — Net picture + the building-identity refinement note (HYPOTHETICAL)
**Building-identity** is the multifamily *refinement input* (it groups permits→buildings); the base
reconciliation runs **without** it (current classification). The projection below — "if the +147 were
harvested and the residual adjudicated" — is **HYPOTHETICAL** and is deliberately **NOT written to the
baseline** (it is not a derived current value).

In [ ]:
print(f'CURRENT (derived):     CO {co} vs city {city} = {co-city}')
print(f'HYPOTHETICAL if +147 harvested: {co}+147 = {co+147} vs {city} = {co+147-city}   # NOT a baseline value')
print('building-identity: refinement input to §6 grouping; base reconciliation does not depend on it.')

## §13 — The gate: derived vs baseline (not hardcoded)
**Design.** The gate compares **DERIVED vs the external timestamped BASELINE file** — never hardcoded
constants. On any mismatch it **DIAGNOSES** (computed value vs baseline value, the v4 sha then-vs-now, the
likely cause from the baseline's `what_would_change_it`) and **HALTs**. **Legitimate change = append a NEW
timestamped baseline** (EVIDENCE-append-only); you NEVER hand-edit a value to make a drifted computation pass.
This is the JN-A anchor-test discipline applied to the reconciliation.

In [ ]:
sha=v4_sha(); fails=[]
for k,spec in BASE['hard_gated'].items():
    got={'co_completions':co,'city_co_total':city,'co_gap':co-city,'bp_issued':pm_lvl}[k]
    ok=got==spec['value']; print(f"  {k:16} derived={got:<7} baseline={spec['value']:<7} {'OK' if ok else 'FAIL'}")
    if not ok: fails.append(f"{k}: derived {got} != baseline {spec['value']} (sha {sha} vs {BASE['v4_sha']}); cause: {spec['what_would_change_it']}; append a new baseline, don't edit.")
assert not fails, 'JN-E GATE HALT:\n'+'\n'.join(fails)
print('GATE PASS — 4 hard figures match', BASE['as_of'], 'baseline.')

## §14 — Assumptions ledger = verifiability ledger (the teachable core)
Each assumption, and **what BREAKS if it is violated** (the failure mode) — so a future reader, or a second
city's analyst, knows the stakes:

| assumption | what it means | what BREAKS if violated |
|---|---|---|
| **oracle-not-source** | CKAN enumerates, never derives | the reconciliation is **circular and void** — you'd be "verifying" the city against itself |
| **count-once** | one building, one count at completion | phased multifamily **double-counts** (over) or **drops** (under) — the −199/+147 errors |
| **null-not-zero** | a missing count is unknown, not 0 | fabricated below-market units; silent under/over-statement of affordability |
| **hold-don't-adopt** | unreproducible city figures are HELD | adopting the +147 makes the number **unverifiable** (oracle-as-source); the gate can no longer protect it |

These four are the lesson: the reconciliation is trustworthy **only** while all four hold. The gate (§13)
enforces the *numbers*; this ledger enforces the *reasoning*.